Change CSV to Parquet

In [49]:
import pandas as pd

df = pd.read_csv("../data/raw/cfpb_raw_2026-02-28_narrative_only.csv")

df.to_parquet("../data/interim/cfpb_raw_2026-02-28_narrative_only.parquet", compression="snappy")

Load Data

In [71]:
import pandas as pd

df = pd.read_parquet("../data/interim/cfpb_raw_2026-02-28_narrative_only.parquet")

print(df.shape)          
print(df.columns)                 

(1404900, 18)
Index(['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue',
       'Consumer complaint narrative', 'Company public response', 'Company',
       'State', 'ZIP code', 'Tags', 'Consumer consent provided?',
       'Submitted via', 'Date sent to company', 'Company response to consumer',
       'Timely response?', 'Consumer disputed?', 'Complaint ID'],
      dtype='str')


Check number of unique complaint categories

In [72]:
print(df["Product"].nunique())

21


Check class balance

In [73]:
df["Product"].value_counts()

Product
Credit reporting or other personal consumer reports                             628016
Credit reporting, credit repair services, or other personal consumer reports    304342
Debt collection                                                                 152342
Checking or savings account                                                      63218
Mortgage                                                                         52591
Money transfer, virtual currency, or money service                               41927
Credit card or prepaid card                                                      41319
Credit card                                                                      41269
Student loan                                                                     22273
Vehicle loan or lease                                                            17960
Credit reporting                                                                 11762
Payday loan, title loan, or persona

Final Department Mapping

In [74]:
product_to_department = {

    # Bank accounts
    "Bank account or service": "Bank accounts",
    "Checking or savings account": "Bank accounts",

    # Consumer loans
    "Consumer Loan": "Consumer loans",
    "Vehicle loan or lease": "Consumer loans",

    # Credit cards (includes prepaid)
    "Credit card": "Card services",
    "Credit card or prepaid card": "Card services",
    "Prepaid card": "Card services",

    # Credit reporting
    "Credit reporting": "Credit reporting",
    "Credit reporting or other personal consumer reports": "Credit reporting",
    "Credit reporting, credit repair services, or other personal consumer reports": "Credit reporting",
    "Debt or credit management": "Credit reporting",

    # Debt collection
    "Debt collection": "Debt collection",

    # Money transfer services
    "Money transfer, virtual currency, or money service": "Money transfer services",
    "Money transfers": "Money transfer services",
    "Virtual currency": "Money transfer services",
    "Other financial service": "Money transfer services",

    # Mortgage
    "Mortgage": "Mortgage",

    # Payday / personal loans
    "Payday loan": "Payday / personal loans",
    "Payday loan, title loan, or personal loan": "Payday / personal loans",
    "Payday loan, title loan, personal loan, or advance loan": "Payday / personal loans",

    # Student loan
    "Student loan": "Student loan"
}

df["Department"] = df["Product"].map(product_to_department)

Check final department distribution

In [75]:
df["Department"].value_counts()

Department
Credit reporting           945844
Debt collection            152342
Card services               86483
Bank accounts               68807
Mortgage                    52591
Money transfer services     42625
Student loan                22273
Consumer loans              21515
Payday / personal loans     12420
Name: count, dtype: int64

Simple rule-based system to classify priority

In [76]:
import re

def assign_priority(text):

    if not isinstance(text, str):
        return "Regular"

    text = text.lower()

    immediate_keywords = [
        "account hacked",
        "card stolen",
        "phishing",
        "scam"
    ]

    same_day_keywords = [
        "identity theft",
        "unauthorized charge",
        "unauthorized transaction",
        "billing error",
        "incorrect charge",
        "double charge",
        "cannot access account",
        "account locked"
    ]

    for word in immediate_keywords:
        if word in text:
            return "Immediate"

    for word in same_day_keywords:
        if word in text:
            return "Same-day"

    return "Regular"

df["Priority"] = df["Consumer complaint narrative"].apply(assign_priority)

In [77]:
df["Priority"].value_counts()

Priority
Regular      1186066
Same-day      193243
Immediate      25591
Name: count, dtype: int64

Check very short and very long narratives

In [78]:
df["n_words"] = df["Consumer complaint narrative"].str.split().str.len()

In [79]:
df["n_words"].describe()

count    1.404900e+06
mean     1.762166e+02
std      2.220826e+02
min      1.000000e+00
25%      6.000000e+01
50%      1.160000e+02
75%      2.110000e+02
max      6.469000e+03
Name: n_words, dtype: float64

In [80]:
sorted_df = df.sort_values("n_words")

In [81]:
sorted_df[["Consumer complaint narrative", "n_words", "Priority"]].head(20)

,Consumer complaint narrative,n_words,Priority
721321,XX/XX/XXXX,1,Regular
912265,XX/XX/XXXX,1,Regular
151266,XX/XX/XXXX,1,Regular
260006,XX/XX/XXXXXX/XX/XXXX,1,Regular
618658,IhaveanaccountonmyfilethatisnotmineIfiledafrau...,1,Regular
1319478,TransunionisreportinganinacurratelistingofChap...,1,Regular
976945,XX/XX/XXXXXXXX,1,Regular
1187403,Letsstartthiscomplaintoffbysayingthatnotonlydo...,1,Regular
111603,XX/XX/XXXX,1,Regular
197068,CapitalOnerefusedtoforceXXXXltoreimbursethemfo...,1,Regular


In [82]:
(df["n_words"] < 5).sum()

np.int64(1204)

In [83]:
sorted_df[["Consumer complaint narrative", "n_words", "Priority"]].tail(20)

,Consumer complaint narrative,n_words,Priority
722133,LexisNexis continues to report an allege bankr...,5858,Regular
894434,XXXX and XXXX continue to report an alleged ba...,5858,Regular
816766,I am writing to formally rescind my [ contract...,5862,Regular
795315,XXXX and Equifax continue to report inaccurate...,5878,Same-day
656325,Navy Federal Credit Union received an applicat...,5879,Same-day
407421,Tranunion and XXXX continue to report inaccura...,5882,Same-day
248682,Lexis Nexis continues to report an alleged ban...,5882,Regular
130624,Can not get a clear title Property because Nat...,5882,Regular
808461,Experian is continuing to report inaccurate in...,5884,Regular
16027,Capital One continues to share my nonpublic fi...,5900,Same-day


In [84]:
(df["n_words"] > 1000).sum()

np.int64(14723)

In [85]:
(df["n_words"] > 1000).mean()

np.float64(0.010479749448359313)

Truncate very long sentences

In [86]:
df["clean_narrative"] = df["Consumer complaint narrative"].str.split().str[:1000].str.join(" ")

Remove very short sentences

In [87]:
df = df[df["n_words"] >= 5]

Detect complaints containing only masks

In [88]:
mask_only = df["clean_narrative"].str.fullmatch(r"[Xx/\s]+")
mask_only.sum()

np.int64(246)

Remove complaints which contain only masks

In [89]:
df = df[~mask_only]

In [90]:
df.shape

(1403450, 22)

Check duplicated narratives

In [91]:
df["clean_narrative"].duplicated().sum()


np.int64(406118)

In [92]:
df.duplicated(subset=["clean_narrative", "Department", "Priority"]).sum()

np.int64(404804)

Remove duplicate narratives

In [93]:
df = df.drop_duplicates(subset=["clean_narrative", "Department", "Priority"])

In [94]:
df.shape

(998646, 22)

In [95]:
df['Priority'].value_counts()

Priority
Regular      848004
Same-day     128218
Immediate     22424
Name: count, dtype: int64

In [96]:
df['Department'].value_counts()

Department
Credit reporting           575006
Debt collection            132349
Card services               81920
Bank accounts               67245
Mortgage                    52548
Money transfer services     33615
Student loan                22184
Consumer loans              21394
Payday / personal loans     12385
Name: count, dtype: int64

Split data

In [97]:
df_model = df[[
    "Complaint ID",
    "clean_narrative",
    "Department",
    "Priority"
]].copy()

In [98]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df_model,
    test_size=0.2,
    stratify=df["Department"],
    random_state=42
)

In [99]:
print(train_df["Department"].value_counts(normalize=True))
print(test_df["Department"].value_counts(normalize=True))

Department
Credit reporting           0.575785
Debt collection            0.132528
Card services              0.082031
Bank accounts              0.067336
Mortgage                   0.052620
Money transfer services    0.033661
Student loan               0.022214
Consumer loans             0.021423
Payday / personal loans    0.012402
Name: proportion, dtype: float64
Department
Credit reporting           0.575787
Debt collection            0.132529
Card services              0.082031
Bank accounts              0.067336
Mortgage                   0.052616
Money transfer services    0.033660
Student loan               0.022215
Consumer loans             0.021424
Payday / personal loans    0.012402
Name: proportion, dtype: float64


In [100]:
print(train_df["Priority"].value_counts(normalize=True))
print(test_df["Priority"].value_counts(normalize=True))

Priority
Regular      0.849265
Same-day     0.128276
Immediate    0.022459
Name: proportion, dtype: float64
Priority
Regular      0.848711
Same-day     0.128854
Immediate    0.022435
Name: proportion, dtype: float64


Save modeling split

In [101]:
train_df.to_parquet(
    "../data/processed/train_complaints.parquet",
    index=False,
    compression="snappy"
)

test_df.to_parquet(
    "../data/processed/test_complaints.parquet",
    index=False,
    compression="snappy"
)

Save the cleaned full dataset

In [102]:
df.to_parquet(
    "../data/interim/cfpb_cleaned.parquet",
    index=False,
    compression="snappy"
)